## NumPy 手写 MLP — 从零理解神经网络内部原理

### 为什么需要这个 Notebook？

PyTorch 的 `loss.backward()` 一行代码就完成了反向传播，但对初学者来说这就像一个「黑盒」。

这个 notebook 用**纯 NumPy** 手动实现：
1. **前向传播**：矩阵乘法 + ReLU + Softmax
2. **交叉熵损失**：手写公式，理解 log 和 sum 的含义
3. **反向传播**：手动推导梯度，逐层回传
4. **参数更新**：梯度下降

### 数学形式

```
z1 = X·W1 + b1       ← 第一层线性变换
a1 = ReLU(z1)        ← 激活
z2 = a1·W2 + b2      ← 第二层线性变换
p  = softmax(z2)     ← 转为概率
loss = -Σ(y·log p)/N ← 交叉熵
```

> 运行完这个 notebook 后，你将对「反向传播到底算了什么」有清晰的认识。

## 1. 导入依赖与数据准备

使用 sklearn 加载 MNIST 子集（10,000 张），加速演示。完整 MNIST 可按需使用。

In [ ]:
# [1. 导入依赖]
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# ==== 中文字体配置 ====
# matplotlib 默认不支持中文，需手动指定支持中文的字体
# 优先级：微软雅黑 > 黑体 > DejaVu Sans（英文回退）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 防止负号显示为方块
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

np.random.seed(0)
print('NumPy 版本:', np.__version__)

In [ ]:
# [1.2 加载 MNIST 数据子集]
# 为加快演示速度，只取前 10000 张
mnist = fetch_openml('mnist_784', version=1, as_frame=False, cache=True, data_home='../data')
X = mnist.data.astype(np.float32) / 255.0  # 归一化到 [0,1]
y = mnist.target.astype(np.int64)

# ======== 超参数集中修改区 ========
N_SUB = 10000          # 使用的样本数（可调大，但训练时间会增加）
# =================================

X = X[:N_SUB]
y = y[:N_SUB]

# 训练/验证划分 (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)
print(f'训练集: {X_train.shape[0]} 样本')
print(f'验证集: {X_val.shape[0]} 样本')
print(f'输入维度: {X_train.shape[1]} (28×28 展平)')

## 2. 模型参数初始化

### 参数形状

| 参数 | 形状 | 说明 |
|------|------|------|
| W1 | (784, 128) | 输入→隐藏层的权重矩阵 |
| b1 | (128,) | 隐藏层偏置 |
| W2 | (128, 10) | 隐藏层→输出层的权重矩阵 |
| b2 | (10,) | 输出层偏置 |

### 为什么要用小随机数初始化？

- 全零初始化 → 所有神经元计算相同，无法学习
- 太大随机数 → 激活值进入饱和区，梯度接近零
- `0.01 × randn` → 小方差高斯分布，适合浅层网络

In [ ]:
# [2. 参数初始化]
D = X_train.shape[1]  # 784 — 输入维度
H = 128               # 隐藏层神经元数（可调）
C = 10                # 类别数（0~9）

def init_params():
    """用小的随机高斯值初始化权重，偏置初始化为 0"""
    W1 = 0.01 * np.random.randn(D, H).astype(np.float32)
    b1 = np.zeros(H, dtype=np.float32)
    W2 = 0.01 * np.random.randn(H, C).astype(np.float32)
    b2 = np.zeros(C, dtype=np.float32)
    return W1, b1, W2, b2

W1, b1, W2, b2 = init_params()

print(f'W1 形状: {W1.shape}  (784 → {H})')
print(f'b1 形状: {b1.shape}')
print(f'W2 形状: {W2.shape}  ({H} → 10)')
print(f'b2 形状: {b2.shape}')
print(f'\n总参数量: {W1.size + b1.size + W2.size + b2.size:,}')

## 3. 前向传播

### 一步步理解

```
z1 = X @ W1 + b1        ← 矩阵乘法: (N,784) @ (784,H) → (N,H)
a1 = max(0, z1)         ← ReLU: 负数变0
z2 = a1 @ W2 + b2       ← (N,H) @ (H,10) → (N,10)
p = softmax(z2)         ← 转为概率分布
```

### Softmax 详解

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

- 将任意实数向量转为概率分布（和为 1）
- 减去 max(z) 是数值稳定技巧，防止 $e^{1000}$ 溢出

In [ ]:
# [3. 前向传播 + 工具函数]
def softmax(z):
    """
    Softmax: 将 logits 转为概率分布

    数值稳定技巧：先减去每行最大值，防止 exp 溢出
    """
    z = z - np.max(z, axis=1, keepdims=True)  # 数值稳定
    expz = np.exp(z)
    return expz / np.sum(expz, axis=1, keepdims=True)


def cross_entropy_loss(p, y_onehot):
    """
    交叉熵损失: -Σ(y_true * log(p)) / N

    p: 预测概率 (N, C)
    y_onehot: 真实标签的 one-hot 编码 (N, C)
    加 eps 避免 log(0)
    """
    N = p.shape[0]
    return -np.sum(y_onehot * np.log(p + 1e-12)) / N


def accuracy(p, y):
    """计算准确率: 预测正确的比例"""
    preds = np.argmax(p, axis=1)
    return np.mean(preds == y)


def one_hot(y, C=10):
    """将整数标签转为 one-hot 编码: 3 → [0,0,0,1,0,0,0,0,0,0]"""
    return np.eye(C)[y]


def forward(X, W1, b1, W2, b2):
    """
    前向传播 — 返回所有中间值，供反向传播使用

    Returns:
        dict: {z1, a1, z2, p} — 中间计算结果
    """
    z1 = X.dot(W1) + b1          # (N, H) — 第一层线性
    a1 = np.maximum(0, z1)       # (N, H) — ReLU 激活
    z2 = a1.dot(W2) + b2         # (N, C) — 第二层线性
    p = softmax(z2)              # (N, C) — 转为概率
    return {'z1': z1, 'a1': a1, 'z2': z2, 'p': p}


# 测试前向传播
test_cache = forward(X_train[:3], W1, b1, W2, b2)
print('测试前向传播 (3 个样本):')
print(f'  z1 形状: {test_cache["z1"].shape}  ← (3, {H})')
print(f'  a1 形状: {test_cache["a1"].shape}  ← (3, {H})')
print(f'  z2 形状: {test_cache["z2"].shape}  ← (3, 10)')
print(f'  p  形状: {test_cache["p"].shape}  ← (3, 10), 每行和为1')
print(f'  概率和: {test_cache["p"].sum(axis=1)}  ← 应全为 1')

## 4. 反向传播（手动推导梯度）

### 链式法则逐层回传

这是本 notebook 最重要的部分！反向传播本质是**链式法则的逐层应用**：

```
Loss → ∂L/∂z2 → ∂L/∂W2, ∂L/∂b2 → ∂L/∂a1 → ∂L/∂z1 → ∂L/∂W1, ∂L/∂b1
```

### 关键梯度公式

| 步骤 | 公式 | 说明 |
|------|------|------|
| Softmax+CE | `dz2 = (p - y_onehot) / N` | Softmax 和交叉熵组合后梯度简洁 |
| dW2 | `a1.T @ dz2` | 权重梯度 = 输入转置 × 输出梯度 |
| db2 | `sum(dz2, axis=0)` | 偏置梯度 = 输出梯度的列求和 |
| ReLU 反向 | `dz1 = da1 * (z1 > 0)` | 正数梯度不变，负数梯度为 0 |
| dW1 | `X.T @ dz1` | 同 dW2 模式 |

In [ ]:
# [4. 反向传播]
def backward(X, y_onehot, params, cache):
    """
    反向传播 — 手动计算所有参数的梯度

    链式法则: Loss → z2 → W2/b2 → a1 → z1 → W1/b1

    Args:
        X: 输入数据 (N, D)
        y_onehot: one-hot 标签 (N, C)
        params: (W1, b1, W2, b2)
        cache: 前向传播的中间结果

    Returns:
        (dW1, db1, dW2, db2) — 各参数的梯度
    """
    W1, b1, W2, b2 = params
    z1 = cache['z1']
    a1 = cache['a1']
    p = cache['p']
    N = X.shape[0]

    # 1. Softmax + CrossEntropy 的组合梯度（简化结果）
    dz2 = (p - y_onehot) / N           # (N, C)

    # 2. 输出层参数梯度
    dW2 = a1.T.dot(dz2)                # (H, C)
    db2 = np.sum(dz2, axis=0)          # (C,)

    # 3. 反向通过 W2 → a1 的梯度
    da1 = dz2.dot(W2.T)                # (N, H)

    # 4. 反向通过 ReLU: 正数梯度为 1，负数梯度为 0
    dz1 = da1 * (z1 > 0)               # (N, H)

    # 5. 隐藏层参数梯度
    dW1 = X.T.dot(dz1)                 # (D, H)
    db1 = np.sum(dz1, axis=0)          # (H,)

    return dW1, db1, dW2, db2


# 验证梯度形状
Xb = X_train[:32]
yb_oh = one_hot(y_train[:32], C)
cache = forward(Xb, W1, b1, W2, b2)
grads = backward(Xb, yb_oh, (W1, b1, W2, b2), cache)
print('梯度形状验证:')
print(f'  dW1: {grads[0].shape} (应为 (784, {H}))')
print(f'  db1: {grads[1].shape} (应为 ({H},))')
print(f'  dW2: {grads[2].shape} (应为 ({H}, 10))')
print(f'  db2: {grads[3].shape} (应为 (10,))')

## 5. 训练循环（小批量 SGD）

### 训练流程

```
for epoch in range(epochs):
    打乱数据
    for batch in batches:
        ① 前向传播 → 获取预测和中间值
        ② 反向传播 → 计算梯度
        ③ 梯度下降 → W = W - lr * dW
    评估 → 打印准确率
```

### 学习率的选择

- 太大 (lr=1.0)：loss 震荡，甚至发散
- 太小 (lr=0.001)：收敛太慢
- 适中 (lr=0.1~0.5)：稳定收敛

In [ ]:
# [5. 训练循环]
def train(X_train, y_train, X_val, y_val, epochs=20, lr=0.1, batch_size=128):
    """
    小批量随机梯度下降 (Mini-batch SGD)

    每次用一个 batch 的数据计算梯度并更新参数

    Args:
        X_train, y_train: 训练数据
        X_val, y_val: 验证数据
        epochs: 训练轮数
        lr: 学习率
        batch_size: 批大小

    Returns:
        history: 包含 train_loss, train_acc, val_loss, val_acc 的字典
    """
    global W1, b1, W2, b2
    n_train = X_train.shape[0]
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        # 每个 epoch 打乱数据顺序
        perm = np.random.permutation(n_train)
        X_shuf = X_train[perm]
        y_shuf = y_train[perm]

        # 按 batch 迭代
        for i in range(0, n_train, batch_size):
            Xb = X_shuf[i:i + batch_size]
            yb = y_shuf[i:i + batch_size]
            yb_oh = one_hot(yb, C)

            # ① 前向传播
            cache = forward(Xb, W1, b1, W2, b2)

            # ② 反向传播
            dW1, db1, dW2, db2 = backward(Xb, yb_oh, (W1, b1, W2, b2), cache)

            # ③ 梯度下降更新
            W1 -= lr * dW1
            b1 -= lr * db1
            W2 -= lr * dW2
            b2 -= lr * db2

        # 每个 epoch 评估一次
        train_cache = forward(X_train[:2000], W1, b1, W2, b2)
        val_cache = forward(X_val, W1, b1, W2, b2)
        train_loss = cross_entropy_loss(train_cache['p'], one_hot(y_train[:2000], C))
        val_loss = cross_entropy_loss(val_cache['p'], one_hot(y_val, C))
        train_acc = accuracy(train_cache['p'], y_train[:2000])
        val_acc = accuracy(val_cache['p'], y_val)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f'Epoch {epoch:2d}: '
              f'train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, '
              f'train_acc={train_acc:.4f}, val_acc={val_acc:.4f}')

    return history


# ======== 超参数集中修改区 ========
EPOCHS = 10
LR = 0.5
BATCH_SIZE = 256
# =================================

# 运行训练
history = train(X_train, y_train, X_val, y_val,
                epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE)

print(f'\n训练完成！最终验证准确率: {history["val_acc"][-1]:.4f}')

## 6. 训练曲线可视化

观察 Loss 是否稳定下降，Train/Val 的差距是否合理。

In [ ]:
# [6. 训练曲线可视化]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：Loss 曲线
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=1.5)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('NumPy MLP Loss 曲线')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图：Accuracy 曲线
axes[1].plot(history['train_acc'], label='Train Acc', linewidth=1.5)
axes[1].plot(history['val_acc'], label='Val Acc', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('NumPy MLP Accuracy 曲线')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 进一步学习

- 对比 PyTorch 版本 `02_mlp_training.ipynb`，感受框架的便利
- 增加隐藏层 → 观察梯度是否越来越小（梯度消失的直观体验）
- 替换 ReLU 为 Sigmoid → 观察 Sigmoid 饱和区的梯度消失